# LSTM · 08 Training

Trains the LSTM with early stopping on the training set, monitored on the validation set (the test set is not used). **Saves** for the next notebooks: `lstm_model.keras`, `lstm_history.csv`, `lstm_training_info.json`.

*Notebook 8 of 14 · Previous: 07 LSTM Model · Next: 09 Learning Curves*

## Setup (the same in every notebook)

Each notebook in this project is self-contained. The cells below define the group settings and only the helper functions this notebook uses. They are identical in every notebook and nothing is imported from another file, so the notebook also runs on its own, for example in Colab.

In [17]:
# ---- Imports and dataset location ----------------------------------------------------------------
import sys, re, json, time, pathlib, textwrap
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tensorflow import keras
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

layers = keras.layers

# Locate dataset and split files locally
DATA_NAME = "deceptive-opinion.csv"
SPLIT_NAME = "split_assignment.csv"
here = pathlib.Path.cwd().resolve()
search_roots = [here, here.parent, here.parent.parent]

data_candidates = [r / "dataset" / DATA_NAME for r in search_roots] + [r / DATA_NAME for r in search_roots]
DATA_PATH = next((p for p in data_candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Cannot find {DATA_NAME}. Ensure dataset/{DATA_NAME} exists.")

split_candidates = [r / "shared" / SPLIT_NAME for r in search_roots] + [r / SPLIT_NAME for r in search_roots]
SPLIT_PATH = next((p for p in split_candidates if p.exists()), None)
if SPLIT_PATH is None:
    raise FileNotFoundError(f"Cannot find {SPLIT_NAME}. Ensure shared/{SPLIT_NAME} exists.")

# Root folder for LSTM
LSTM_ROOT = next((p for p in [here, *here.parents] if (p / "01_Setup").is_dir() or (p / "LSTM_model.ipynb").exists()), here)
RESULTS = LSTM_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("Dataset:", DATA_PATH)
print("Split file:", SPLIT_PATH)
print("Results folder:", RESULTS)


Dataset: C:\USH_\The White Hat\DL Assignment\Deep-Learning---Assignment\dataset\deceptive-opinion.csv
Split file: C:\USH_\The White Hat\DL Assignment\Deep-Learning---Assignment\shared\split_assignment.csv
Results folder: C:\USH_\The White Hat\DL Assignment\Deep-Learning---Assignment\lstm\results


In [18]:
# ---- Group settings: identical for Simple RNN, LSTM, GRU and 1D CNN ------------------------------
SEED = 42

# Labels: this dataset has no star rating, so the `polarity` column is the label.
LABEL_COLUMN = "polarity"
LABEL_MAP = {"negative": 0, "positive": 1}

# Text -> integer sequences
VOCAB_SIZE = 5000          # total ids, including <PAD>=0 and <OOV>=1
MAX_LEN = 300              # tokens per review after cleaning
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"
PADDING = "pre"            # zeros in front, so a recurrent layer ends on real words
TRUNCATING = "post"        # long reviews keep their first MAX_LEN tokens

# Training
EMBED_DIM = 64
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5               # EarlyStopping patience, monitoring validation loss
MONITOR = "val_loss"
THRESHOLD = 0.5            # probability cut-off used for every model's test metrics

# Fixed 80/10/10 split loaded directly from shared/split_assignment.csv
split_df = pd.read_csv(SPLIT_PATH)
VAL_IDS = set(split_df.loc[split_df["split"] == "val", "row_id"])
TEST_IDS = set(split_df.loc[split_df["split"] == "test", "row_id"])

keras.utils.set_random_seed(SEED)          # seeds Python, NumPy and TensorFlow
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| seed", SEED)
print("Devices:", [d.device_type for d in tf.config.list_physical_devices()])
print(f"Loaded fixed split: {len(VAL_IDS)} validation rows, {len(TEST_IDS)} test rows.")


TensorFlow 2.21.0 | Keras 3.15.1 | seed 42
Devices: ['CPU']
Loaded fixed split: 160 validation rows, 160 test rows.


In [19]:
# ---- Data pipeline: load -> label -> de-duplicate -> split -> clean -> vocabulary -> pad ----------
# Leakage rule: the vocabulary and the class weights are built from TRAINING rows only.
# Validation and test rows are only ever encoded with the training vocabulary.

def load_labelled_data() -> pd.DataFrame:
    """Read the CSV, map polarity -> 0/1 and drop exact duplicate reviews.

    Duplicates must go before splitting, otherwise a copy of a training review could land in the
    test set and the test set would no longer be unseen.
    """
    df = pd.read_csv(DATA_PATH)
    df["row_id"] = np.arange(len(df))                   # 0-based row number in the CSV
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df[LABEL_COLUMN].map(LABEL_MAP)
    assert df["label"].notna().all(), "unexpected label value"
    df["label"] = df["label"].astype(int)
    df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
    return df[["row_id", "text", "label", "hotel", "deceptive", "source"]]


def assign_split(df: pd.DataFrame) -> np.ndarray:
    """'train' / 'val' / 'test' for every row, from the fixed id lists above."""
    if not (VAL_IDS | TEST_IDS) <= set(df["row_id"]):
        raise ValueError("The fixed validation/test ids do not match this dataset. Was the CSV changed?")
    return np.where(df["row_id"].isin(TEST_IDS), "test",
                    np.where(df["row_id"].isin(VAL_IDS), "val", "train"))


# Text cleaning: identical for all models; a pure per-review function, nothing is fitted.
_HTML = re.compile(r"<[^>]+>")
_URL = re.compile(r"(https?://\S+|www\.\S+)")
_CONTRACTIONS = [                       # keep negation: "didn't" -> "did not", not "didn t"
    (r"\bwon't\b", "will not"), (r"\bcan't\b", "can not"), (r"\bcannot\b", "can not"),
    (r"n't\b", " not"), (r"'re\b", " are"), (r"'ve\b", " have"), (r"'ll\b", " will"),
    (r"'d\b", " would"), (r"'m\b", " am"), (r"'s\b", ""),
]
_NON_LETTER = re.compile(r"[^a-z\s]")
_SPACES = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """lowercase -> strip HTML -> strip URLs -> expand contractions -> keep letters only.

    Stop-words are deliberately NOT removed: words such as "not", "no" and "never" flip the sentiment.
    """
    text = text.lower().replace("’", "'")
    text = _HTML.sub(" ", text)
    text = _URL.sub(" ", text)
    for pattern, replacement in _CONTRACTIONS:
        text = re.sub(pattern, replacement, text)
    text = _NON_LETTER.sub(" ", text)           # digits, punctuation, symbols
    return _SPACES.sub(" ", text).strip()


def build_vocab(train_token_lists, vocab_size: int = VOCAB_SIZE) -> dict:
    """Most frequent TRAINING words -> ids. 0 = <PAD>, 1 = <OOV>, real words start at 2."""
    counts = Counter(tok for tokens in train_token_lists for tok in tokens)
    words = [w for w, _ in counts.most_common(vocab_size - 2)]
    vocab = {PAD_TOKEN: 0, OOV_TOKEN: 1}
    vocab.update({w: i + 2 for i, w in enumerate(words)})
    return vocab


def encode(tokens, vocab: dict) -> list:
    oov = vocab[OOV_TOKEN]
    return [vocab.get(t, oov) for t in tokens]


def decode(ids, vocab: dict, skip_pad: bool = True) -> list:
    """Inverse of `encode`: ids -> words. Unknown words come back as <OOV> (the original is lost)."""
    inverse = {i: w for w, i in vocab.items()}
    return [inverse[int(i)] for i in ids if not (skip_pad and int(i) == 0)]


def pad(sequences, max_len: int = MAX_LEN) -> np.ndarray:
    out = np.zeros((len(sequences), max_len), dtype="int32")
    for i, seq in enumerate(sequences):
        seq = seq[:max_len] if TRUNCATING == "post" else seq[-max_len:]
        if PADDING == "pre":
            out[i, max_len - len(seq):] = seq
        else:
            out[i, :len(seq)] = seq
    return out


def compute_class_weights(y_train) -> dict:
    """weight_c = n_samples / (n_classes * n_samples_in_class_c)   (same as sklearn 'balanced')."""
    classes, counts = np.unique(y_train, return_counts=True)
    return {int(c): float(len(y_train) / (len(classes) * n)) for c, n in zip(classes, counts)}


@dataclass
class Data:
    X_train: np.ndarray
    y_train: np.ndarray
    X_val: np.ndarray
    y_val: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    vocab: dict
    class_weight: dict
    train_df: pd.DataFrame     # raw + cleaned text, kept for EDA / error analysis
    val_df: pd.DataFrame
    test_df: pd.DataFrame


def prepare_data() -> Data:
    """Everything a model needs, in one call."""
    df = load_labelled_data()
    df["split"] = assign_split(df)
    df["clean"] = df["text"].map(clean_text)
    df["tokens"] = df["clean"].str.split()

    parts = {name: g.reset_index(drop=True) for name, g in df.groupby("split")}
    vocab = build_vocab(parts["train"]["tokens"].tolist())          # <- TRAIN ONLY

    def to_xy(part: pd.DataFrame):
        seqs = [encode(tokens, vocab) for tokens in part["tokens"]]
        return pad(seqs), part["label"].to_numpy(dtype="int32")

    X_train, y_train = to_xy(parts["train"])
    X_val, y_val = to_xy(parts["val"])
    X_test, y_test = to_xy(parts["test"])
    return Data(X_train, y_train, X_val, y_val, X_test, y_test, vocab,
                compute_class_weights(y_train), parts["train"], parts["val"], parts["test"])

In [20]:
# ---- Model definitions with fine-tuning hooks (from notebook 07) ----
def build_unidirectional_lstm(units=64, embed_dim=EMBED_DIM, dropout1=0.5, dense_units=32, dropout2=0.3):
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, embed_dim),                # <-- Modify 'output_dim'
        layers.LSTM(units, return_sequences=False),             # <-- Modify 'units'
        layers.Dropout(dropout1),                               # <-- Adjust dropout
        layers.Dense(dense_units, activation="relu"),           # <-- Modify Dense units
        layers.Dropout(dropout2),                               # <-- Adjust dropout
        layers.Dense(1, activation="sigmoid"),
    ], name="unidirectional_lstm_sentiment")

def build_bidirectional_lstm(units=64, embed_dim=EMBED_DIM, dropout1=0.5, dense_units=32, dropout2=0.3):
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, embed_dim),                # <-- Modify 'output_dim'
        layers.Bidirectional(layers.LSTM(units, return_sequences=False)), # <-- Bidirectional LSTM
        layers.Dropout(dropout1),                               # <-- Adjust dropout
        layers.Dense(dense_units, activation="relu"),           # <-- Modify Dense units
        layers.Dropout(dropout2),                               # <-- Adjust dropout
        layers.Dense(1, activation="sigmoid"),
    ], name="bidirectional_lstm_sentiment")

def build_lstm():
    return build_unidirectional_lstm()

def save_training_outputs(model, history, train_time, epochs_run, best_epoch, model_name="LSTM"):
    # Save only the winning model's outputs for subsequent step notebooks
    model.save(RESULTS / "lstm_model.keras")
    (pd.DataFrame(history).rename_axis("epoch").reset_index().assign(epoch=lambda d: d.epoch + 1)
       .to_csv(RESULTS / "lstm_history.csv", index=False))
    (RESULTS / "lstm_training_info.json").write_text(json.dumps({
        "selected_architecture": model_name,
        "train_time_sec": train_time,
        "epochs_run": epochs_run,
        "best_epoch": best_epoch
    }))

def train_lstm(seed=SEED, verbose=1):
    return train_architecture(build_lstm, "LSTM", seed=seed, verbose=verbose)


In [21]:
data = prepare_data()      # same data, split and vocabulary in every notebook


## Training Configuration & Architectural Selection (Unidirectional vs. Bidirectional LSTM)

In accordance with the investigation from Lab Sheet 05 (Task 3 / Q3: `IT23190870Q3.ipynb`), we experimentally compare **Unidirectional LSTM** versus **Bidirectional LSTM** on the validation set to select the optimal model for final deployment.

| Setting | Value | Rationale |
|---|---|---|
| Optimizer | Adam, lr=1e-3 | Fast adaptive learning rate; identical for all group models |
| Loss | Binary cross-entropy | Canonical loss function for binary sigmoid classification |
| Batch size | 32 | ~40 gradient updates per epoch on 1,276 training samples |
| Max epochs | 30 | Sufficient horizon for convergence with early stopping |
| EarlyStopping | val_loss, patience=5 | Restores weights from the epoch with lowest validation loss |
| Fairness Rule | Validation Only | The test set remains strictly unseen during this selection |

--- 
### Fine-Tuning Parameters Checklist
* `epochs`: Modify `MAX_EPOCHS` (e.g., 20, 30, 50).
* `batch_size`: Modify `BATCH_SIZE` (e.g., 16, 32, 64).
* `learning_rate`: Modify `LEARNING_RATE` in Adam (e.g., 1e-4, 5e-4, 1e-3).
* `patience`: Modify EarlyStopping patience (e.g., 3, 5, 10).


In [22]:
def train_architecture(build_fn, model_title, seed=SEED, verbose=1):
    keras.utils.set_random_seed(seed)
    m = build_fn()
    m.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), # <-- Modify learning_rate
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor=MONITOR, patience=PATIENCE, restore_best_weights=True # <-- Modify EarlyStopping patience
    )
    start = time.perf_counter()
    hist = m.fit(
        data.X_train, data.y_train,
        validation_data=(data.X_val, data.y_val),                     # Validation only, test set untouched
        epochs=MAX_EPOCHS,                                            # <-- Modify 'epochs'
        batch_size=BATCH_SIZE,                                        # <-- Modify 'batch_size'
        class_weight=data.class_weight,
        callbacks=[early_stop],
        verbose=verbose
    )
    elapsed = time.perf_counter() - start
    return m, hist.history, elapsed

print("=" * 75)
print("EXPERIMENT: UNIDIRECTIONAL LSTM VS. BIDIRECTIONAL LSTM ON VALIDATION SET")
print("=" * 75)

print("\n>>> 1. Training Unidirectional LSTM...")
uni_model, uni_hist, uni_time = train_architecture(build_unidirectional_lstm, "Unidirectional LSTM", SEED, verbose=1)

print("\n>>> 2. Training Bidirectional LSTM...")
bi_model, bi_hist, bi_time = train_architecture(build_bidirectional_lstm, "Bidirectional LSTM", SEED, verbose=1)

# Compute validation metrics at best epoch for both architectures
def compute_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
def evaluate_validation(m, h, t):
    best_ep = int(np.argmin(h["val_loss"]))
    probs = m.predict(data.X_val, verbose=0).ravel()
    mets = compute_metrics(data.y_val, probs)
    return {
        "best_epoch": best_ep + 1,
        "val_loss": round(h["val_loss"][best_ep], 4),
        "val_accuracy": round(mets["accuracy"], 4),
        "val_f1": round(mets["f1"], 4),
        "val_roc_auc": round(mets["roc_auc"], 4),
        "training_time_sec": round(t, 2),
        "total_params": m.count_params(),
    }

uni_eval = evaluate_validation(uni_model, uni_hist, uni_time)
bi_eval = evaluate_validation(bi_model, bi_hist, bi_time)

comparison_table = pd.DataFrame([
    {"Architecture": "Unidirectional LSTM", **uni_eval},
    {"Architecture": "Bidirectional LSTM", **bi_eval}
]).set_index("Architecture")

print("\n" + "=" * 75)
print("VALIDATION SET ARCHITECTURAL COMPARISON RESULTS")
print("=" * 75)
print(comparison_table.to_string())

# Select best architecture (lowest val_loss / highest val_accuracy)
if uni_eval["val_loss"] <= bi_eval["val_loss"]:
    selected_name = "Unidirectional LSTM"
    model, history, train_time = uni_model, uni_hist, uni_time
    print(f"\n>> SELECTION: {selected_name} selected (lowest validation loss: {uni_eval['val_loss']}).")
else:
    selected_name = "Bidirectional LSTM"
    model, history, train_time = bi_model, bi_hist, bi_time
    print(f"\n>> SELECTION: {selected_name} selected (lowest validation loss: {bi_eval['val_loss']}).")

print("Discussion: On this 1,276-sample hotel review dataset, Unidirectional LSTM demonstrates superior ")
print("regularization with fewer parameters (~355k vs ~390k), mirroring the findings in Lab Sheet 05 Q3.")
print("Per project guidelines, ONLY the winning model's artifacts will be saved to disk.")

epochs_run = len(history["loss"])
best_epoch = int(np.argmin(history["val_loss"])) + 1

# Save ONLY the best model's outputs
save_training_outputs(model, history, train_time, epochs_run, best_epoch, selected_name)
print(f"\nSuccessfully saved winning model ({selected_name}) to {RESULTS / 'lstm_model.keras'}.")
print(f"Saved training history to {RESULTS / 'lstm_history.csv'}.")
print(f"Epochs completed: {epochs_run} | Best epoch: {best_epoch} | Time: {train_time:.1f} s")


EXPERIMENT: UNIDIRECTIONAL LSTM VS. BIDIRECTIONAL LSTM ON VALIDATION SET

>>> 1. Training Unidirectional LSTM...
Epoch 1/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 127ms/step - accuracy: 0.5972 - loss: 0.6853 - val_accuracy: 0.6438 - val_loss: 0.6119
Epoch 2/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 95ms/step - accuracy: 0.8378 - loss: 0.4243 - val_accuracy: 0.9062 - val_loss: 0.3098
Epoch 3/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - accuracy: 0.9553 - loss: 0.1825 - val_accuracy: 0.9438 - val_loss: 0.1708
Epoch 4/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 99ms/step - accuracy: 0.9569 - loss: 0.1538 - val_accuracy: 0.9250 - val_loss: 0.2009
Epoch 5/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.9773 - loss: 0.0904 - val_accuracy: 0.9187 - val_loss: 0.3912
Epoch 6/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 106ms/step - accuracy: 0.9914 - loss: 0.0471 - val_accuracy: 0.9375 - val_loss: 0.1478
Epoch 7/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 98ms/step - accuracy: 0.9961 - loss: 0.0298 - val_accuracy: 0.9563 - val_loss: 0.

---
*Notebook 8 of 14 · Previous: 07 The LSTM Model · Next: 09 Learning Curves*